# Best-of-N Speed Benchmark Across Models

Benchmark Best-of-N generation speed on the same set of MATH
problems across multiple large language models.

Each model is loaded with vLLM, warmed up, timed for
`num_trials` runs, and then unloaded before the next model is
tested. This keeps the comparison focused on model-level
throughput under the same benchmark settings.

Use this notebook to compare speed across model families or model
sizes. Use `benchmark_speed_bon_quant_v1.ipynb` for the separate
quantization sweep.

## Setup

In [ ]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL + 1)

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")

import statistics

from utils.configs import GenConfig

from utils.load_data import load_data_hf
from unittests.notebook_utils import benchmark_bon_speed

In [ ]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets'

ds_split = "test"
ds_dir = f"{base_dir}/prm800k/math_splits"


# Models to benchmark — same prompts, same config; only this varies.
# Each entry is a model config dict for benchmark_bon_speed (same shape
# as the quant notebook; name auto-derived, fp16/unquantized here).
def _cfg(subdir):
    return {"model_dir": os.path.join(base_dir, subdir)}


model_configs = [
    _cfg("Llama3.2-1B-Instruct"),
    _cfg("Llama3.2-3B-Instruct"),
    # _cfg("Llama3.2-7B-Instruct"),
    _cfg("Qwen2.5-Math-1.5B-Instruct"),
    _cfg("Qwen2.5-Math-7B-Instruct"),
]

In [ ]:
# Best-of-N search params.
# Aligned with benchmark_speed_bon_quant_v1 so the two notebooks run
# the SAME workload (n, gmu, questions, trials) and their numbers are
# directly comparable.
config = GenConfig()
config.agg_strategy = 'last'
config.temperature = 0.8
config.max_tokens = 2048
config.n = 32
config.filter_duplicates = True
config.date_string = "Aug 1 2025"
config.seed = 0

# Benchmark knobs (kept identical across both speed notebooks)
level = 4                          # MATH difficulty level
MAX_QUESTIONS = 5                  # cap on questions per benchmark
num_trials = 2                     # timed runs per model
warmup = 1                         # untimed warmup runs per model
llm_gpu_memory_utilization = 0.3

In [4]:
dataset = load_data_hf(ds_dir, ds_split=ds_split, level=level)
num_questions = min(len(dataset), MAX_QUESTIONS)
batch_of_questions = [dataset[i]['problem'] for i in range(num_questions)]
print(f"num_questions = {num_questions}")

num_questions = 10


## Run benchmark

One model at a time; teardown between iterations frees the vLLM
engine before the next is loaded.

In [ ]:
results = []
for cfg in model_configs:
    name, times = benchmark_bon_speed(
        cfg, config, batch_of_questions, num_trials,
        gpu_memory_utilization=llm_gpu_memory_utilization,
        warmup=warmup,
    )
    results.append((name, times))

## Summary

In [ ]:
print(
    f"=== Summary (level={level}, "
    f"n_questions={num_questions}, n_trials={num_trials}) ==="
)
header = (
    f"{'model':<25}{'mean s/trial':>14}{'std':>8}{'s/question':>14}"
)
print(header)
print('-' * len(header))
for name, times in results:
    mean = statistics.mean(times)
    std = statistics.stdev(times) if len(times) > 1 else 0.0
    print(
        f"{name:<25}{mean:>14.2f}{std:>8.2f}{mean/num_questions:>14.4f}"
    )